# Week 4 · Day 3 — Analytical SQL: window functions, `QUALIFY` & CTEs

*The questions a spreadsheet chokes on — rank within each group, running totals, top-N-per-category — are one line of SQL in a warehouse.*

**By the end you'll have shipped:** a **top-item-per-store** report built with a **window function** and **`QUALIFY`**, plus a **running-revenue** column and a readable **CTE** — the analytics a warehouse is *built* for.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M4 · Data & Snowflake (Week 4) |
| **Prerequisites** | W3D2 (`GROUP BY`), W4D1–D2 |
| **Est. time** | ~30 min |
| **Capstone slice** | The **analytics layer** — ranking and trending matters for a partner view |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — real Snowflake syntax (incl. `QUALIFY`), run on DuckDB |

### 🎯 Learning objectives

By the end you'll be able to:
- Use a **CTE** (`WITH ... AS`) to name a subquery and keep SQL readable.
- Write **`CASE WHEN`** — SQL's if/else — to bucket rows.
- Use a **window function** (`ROW_NUMBER`, `RANK`, running `SUM`) with `OVER (PARTITION BY ... ORDER BY ...)`.
- Filter on a window result with **`QUALIFY`** (Snowflake's superpower) to get **top-N per group**.
- Explain how a window differs from `GROUP BY` (it keeps every row).

### ⚖️ Why it matters

Partners don't ask "total billings" — they ask *"the **largest** matter per practice area,"* *"each client's **most recent** matter,"* *"**running** revenue this quarter."* Those are **within-group ranking** and **cumulative** questions. `GROUP BY` can't answer them (it collapses rows); **window functions** can. This is the SQL that turns a warehouse into an analytics engine — and it's exactly what a *Matter Intelligence* dashboard runs.

### ⚙️ Setup

The usual `run_sql` helper over `coffee_orders`. Everything here is standard Snowflake SQL; offline it runs on DuckDB, which supports the same window syntax **and `QUALIFY`** (something SQLite can't do — a big reason we chose DuckDB as the stand-in).

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {
    'coffee_orders': [{'order_id': 'O-5001', 'date': '2026-03-07', 'item': 'Cappuccino', 'size': 'S', 'category': 'Espresso Drink', 'price': 3.75, 'payment': 'Cash', 'store': 'Downtown'}, {'order_id': 'O-5002', 'date': '2026-03-07', 'item': 'Mocha', 'size': 'S', 'category': 'Espresso Drink', 'price': 4.5, 'payment': 'Card', 'store': 'Airport'}, {'order_id': 'O-5003', 'date': '2026-03-05', 'item': 'Cappuccino', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.25, 'payment': 'Card', 'store': 'Downtown'}, {'order_id': 'O-5004', 'date': '2026-03-05', 'item': 'Croissant', 'size': 'M', 'category': 'Food', 'price': 3.25, 'payment': 'App', 'store': 'Uptown'}, {'order_id': 'O-5005', 'date': '2026-03-06', 'item': 'Latte', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.5, 'payment': 'App', 'store': 'Downtown'}],
}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

### 1 · CTEs — name a subquery with `WITH`

A **CTE** (Common Table Expression) is a temporary, named result you define at the top with `WITH`, then use like a table. It turns nested, unreadable SQL into a top-to-bottom story. Think of it as a **defined term** in a contract: define `store_totals` once, refer to it cleanly after.

In [ ]:
run_sql("""
    WITH store_totals AS (
        SELECT store, ROUND(SUM(price), 2) AS revenue
        FROM coffee_orders
        GROUP BY store
    )
    SELECT *
    FROM store_totals
    WHERE revenue > 10
    ORDER BY revenue DESC
""")

**What just happened:** the `WITH store_totals AS (...)` block computed per-store revenue once; the main query then filtered it. Same result as a subquery, far easier to read — and you can define several CTEs, each building on the last.

### 2 · `CASE WHEN` — SQL's if/else

**`CASE WHEN condition THEN value ... ELSE value END`** classifies rows inline — the SQL twin of an if/else, and of pandas `.apply`. Here we bucket each order into a price tier.

In [ ]:
run_sql("""
    SELECT item, price,
           CASE
               WHEN price >= 5      THEN 'premium'
               WHEN price >= 3.5    THEN 'standard'
               ELSE 'value'
           END AS price_tier
    FROM coffee_orders
    ORDER BY price DESC
    LIMIT 6
""")

### 3 · Window functions — rank **without** collapsing rows

`GROUP BY` returns *one row per group*. A **window function** computes across a group but **keeps every row**, adding the calculation as a new column. The engine is `OVER (PARTITION BY ... ORDER BY ...)`:

- **`PARTITION BY store`** = "do this separately within each store" (like `GROUP BY`, but non-collapsing).
- **`ORDER BY price DESC`** = the order the function walks the rows in.

**`ROW_NUMBER()`** numbers rows 1, 2, 3… within each partition — so #1 is each store's priciest order.

In [ ]:
run_sql("""
    SELECT store, item, price,
           ROW_NUMBER() OVER (PARTITION BY store ORDER BY price DESC) AS rank_in_store
    FROM coffee_orders
    ORDER BY store, rank_in_store
""")

**What just happened:** every order kept its row, but each now carries its **rank within its store**. Notice we didn't lose any data — that's the difference from `GROUP BY`.

### 4 · `QUALIFY` — filter on the window result (top-N per group)

You have a rank column — now you want *only rank 1 per store*. You can't put a window function in `WHERE` (it runs too early). **`QUALIFY`** is Snowflake's clause for exactly this: it filters on a window function's result, the way `HAVING` filters on an aggregate.

In [ ]:
# the single priciest order at each store — top-1-per-group in one query
run_sql("""
    SELECT store, item, price
    FROM coffee_orders
    QUALIFY ROW_NUMBER() OVER (PARTITION BY store ORDER BY price DESC) = 1
    ORDER BY price DESC
""")

**What just happened:** `QUALIFY ... = 1` kept only each store's top-ranked row — a "top-N per group" report in a single, clean statement. Without `QUALIFY` you'd need a CTE + subquery; this is why analysts love it.

### 5 · Running totals — `SUM(...) OVER (ORDER BY ...)`

Add an `ORDER BY` inside `OVER` *without* a `PARTITION BY` and an aggregate becomes **cumulative** — a running total. Great for "revenue to date."

In [ ]:
run_sql("""
    SELECT date, item, price,
           ROUND(SUM(price) OVER (ORDER BY date, order_id), 2) AS running_revenue
    FROM coffee_orders
    ORDER BY date, order_id
""")

> **`Go Deeper 🔧` — `RANK` vs `ROW_NUMBER`, and per-group shares.**
> - **`ROW_NUMBER`** always gives distinct 1,2,3 (ties broken arbitrarily). **`RANK`** gives ties the *same* number and then skips (1,1,3). **`DENSE_RANK`** ties without skipping (1,1,2).
> - Windows can compute a **share of group**: `price / SUM(price) OVER (PARTITION BY store)` = each order's fraction of its store's revenue.

In [ ]:
run_sql("""
    SELECT store, item, price,
           RANK()       OVER (PARTITION BY store ORDER BY price DESC) AS rnk,
           ROUND(100 * price / SUM(price) OVER (PARTITION BY store), 1) AS pct_of_store
    FROM coffee_orders
    ORDER BY store, rnk
""")

> **`Common pitfalls ⚠️`**
>
> - **A window function can't go in `WHERE`** — use **`QUALIFY`** (Snowflake) to filter on it.
> - **`GROUP BY` collapses; `OVER` keeps every row.** Pick by whether you want one row per group or a new column on every row.
> - **`PARTITION BY` ≠ `GROUP BY`** — same "per group" idea, but the window adds a column instead of reducing rows.
> - Always give `OVER` an **`ORDER BY`** for ranking/running functions, or the order is undefined.

### ✍️ Your turn

In [ ]:
# TODO 1: number each store's orders from cheapest to priciest (ROW_NUMBER, ORDER BY price ASC)

# TODO 2: keep only the CHEAPEST order per store (QUALIFY ... = 1)

# TODO 3: add a running COUNT of orders over date+order_id (COUNT(*) OVER (ORDER BY ...))


<details><summary>✅ Show solution</summary>

```python
# 1
run_sql("""
    SELECT store, item, price,
           ROW_NUMBER() OVER (PARTITION BY store ORDER BY price ASC) AS n
    FROM coffee_orders
    ORDER BY store, n
""")

# 2
run_sql("""
    SELECT store, item, price
    FROM coffee_orders
    QUALIFY ROW_NUMBER() OVER (PARTITION BY store ORDER BY price ASC) = 1
    ORDER BY price
""")

# 3
run_sql("""
    SELECT date, order_id, item,
           COUNT(*) OVER (ORDER BY date, order_id) AS orders_so_far
    FROM coffee_orders
    ORDER BY date, order_id
""")
```
</details>

### 🚀 Build the artifact — a "best seller per store" report

Combine a **CTE** (compute per-item revenue per store) with a **window + `QUALIFY`** (keep the top item in each store). This is a genuinely useful report that's clumsy in a spreadsheet and one query here.

In [ ]:
def best_seller_per_store() -> pd.DataFrame:
    """Each store's top item by revenue — CTE + window + QUALIFY."""
    return run_sql("""
        WITH item_rev AS (
            SELECT store, item,
                   ROUND(SUM(price), 2) AS revenue,
                   COUNT(*)             AS orders
            FROM coffee_orders
            GROUP BY store, item
        )
        SELECT store, item, revenue, orders
        FROM item_rev
        QUALIFY ROW_NUMBER() OVER (PARTITION BY store ORDER BY revenue DESC) = 1
        ORDER BY revenue DESC
    """)

best_seller_per_store()

> **🔗 Your world — from coffee to matters.** These are the partner-dashboard queries:
>
> ```sql
> -- the largest matter in each practice area
> SELECT practice_area, matter_id, client, amount_billed
> FROM matters
> QUALIFY ROW_NUMBER() OVER (PARTITION BY practice_area ORDER BY amount_billed DESC) = 1;
>
> -- each client's most recent matter
> SELECT client, matter_id, open_date
> FROM matters
> QUALIFY ROW_NUMBER() OVER (PARTITION BY client ORDER BY open_date DESC) = 1;
> ```
>
> "Biggest matter per area," "latest matter per client," "running billings this quarter" — window functions answer the questions partners actually ask.

### 📝 Recap — what you shipped

- **CTEs** (`WITH`) name subqueries so complex SQL reads top-to-bottom.
- **`CASE WHEN`** is SQL's if/else for bucketing rows.
- **Window functions** (`ROW_NUMBER`, `RANK`, running `SUM`) compute across a group **without collapsing rows**, via `OVER (PARTITION BY ... ORDER BY ...)`.
- **`QUALIFY`** filters on a window result — the clean way to get **top-N per group**.
- **Artifact:** `best_seller_per_store()` — a CTE + window + `QUALIFY` report.

### 🧠 Check your understanding

1. How does a **window function** differ from `GROUP BY`?
2. Why can't you filter a window function in `WHERE` — and what do you use instead?
3. What's the difference between `ROW_NUMBER` and `RANK` on ties?
4. What does `SUM(price) OVER (ORDER BY date)` compute?

<details><summary>✅ Answers</summary>

1. `GROUP BY` returns **one row per group**; a window function computes across the group but **keeps every row**, adding the result as a new column.
2. `WHERE` runs **before** window functions are computed; use **`QUALIFY`** to filter on a window result (like `HAVING` for aggregates).
3. `ROW_NUMBER` gives every row a **distinct** number even on ties; `RANK` gives tied rows the **same** number and then **skips** (1,1,3).
4. A **running (cumulative) total** of `price` in date order.
</details>

### ➡️ Next up — Week 4, Day 4: Cortex — an LLM **inside** Snowflake

You can store, load, and analyze matters. The finale: **summarize and classify** them *without leaving SQL*, using Snowflake **Cortex** (`SUMMARIZE`, `CLASSIFY_TEXT`, `COMPLETE`) — the exact *Matter Intelligence* capstone slice, and the bridge to the Claude module.

*Same toolkit; Cortex is mocked offline, just like the Claude lessons mock the API.*

### 📖 Reference & glossary

| Term | Plain meaning |
|---|---|
| **CTE (`WITH`)** | a named, reusable subquery |
| **`CASE WHEN`** | SQL if/else that labels rows |
| **Window function** | a calc across a group that keeps every row |
| **`OVER (PARTITION BY ... ORDER BY ...)`** | the window's group + order |
| **`ROW_NUMBER` / `RANK` / `DENSE_RANK`** | numbering / ranking (differ on ties) |
| **`QUALIFY`** | filter on a window function's result |
| **Running total** | cumulative aggregate via `OVER (ORDER BY ...)` |

**Docs:** Snowflake window functions — https://docs.snowflake.com/en/sql-reference/functions-analytic · `QUALIFY` — https://docs.snowflake.com/en/sql-reference/constructs/qualify

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*